# 00: Titanic Kaggle Main Workflow

**Theme:** Python transforms raw historical data into insight, prediction, and learning.

This is the main notebook and the most important notebook of the project. It details the complete Kaggle Titanic workflow using the official Kaggle dataset files.

### Kaggle Dataset Structure
* **`train.csv`**: Contains passenger labels (`Survived` column) and is used for EDA, training, and validation.
* **`test.csv`**: Unseen passenger data without target labels. Used for generating final predictions for submission.
* **`gender_submission.csv`**: Baseline/example submission format. Do not use this as ground-truth labels.


## 1. Environment & Library Setup
Declare all imports and configurations.

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


## 2. Dataset Loading and Audit
Load the dataset from the local `titanic/` folder or fallback Google Colab paths.


In [ ]:
# Define search paths
possible_paths = [
    "../titanic/",
    "./titanic/",
    "/content/",
    "/content/drive/MyDrive/titanic/"
]

train, test, gender_sub = None, None, None

for base_path in possible_paths:
    if os.path.exists(os.path.join(base_path, "train.csv")):
        train = pd.read_csv(os.path.join(base_path, "train.csv"))
        test = pd.read_csv(os.path.join(base_path, "test.csv"))
        gender_sub = pd.read_csv(os.path.join(base_path, "gender_submission.csv"))
        print(f"Loaded successfully from: {base_path}")
        break

if train is None:
    raise FileNotFoundError("Could not locate Kaggle Titanic dataset files!")

# Ingestion check
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Gender Submission shape:", gender_sub.shape)

# Validation checks
assert "Survived" in train.columns, "train.csv must contain 'Survived'!"
assert "Survived" not in test.columns, "test.csv must NOT contain 'Survived'!"
assert list(gender_sub.columns) == ["PassengerId", "Survived"], "gender_submission.csv format mismatch!"
print("✓ Dataset audit checks passed!")


## 3. Basic Titanic Questions
Explore passenger statistics directly with Pandas.


In [ ]:
print("1. How many passengers survived and died?")
                survived_count = train["Survived"].value_counts()
                print(f"Survived: {survived_count.get(1, 0)}, Died: {survived_count.get(0, 0)}")

                print("
2. Gender count:")
                gender_count = train["Sex"].value_counts()
                print(gender_count)

                print("
3. How many males and females survived?")
                gender_survived = train.groupby("Sex")["Survived"].sum()
                print(gender_survived)

                print("
4. What is the survival rate by gender?")
                gender_survival_rate = train.groupby("Sex")["Survived"].mean()
                print(gender_survival_rate)

                print("
5. Did passenger class affect survival?")
                class_survival_rate = train.groupby("Pclass")["Survived"].mean()
                print(class_survival_rate)

                print("
6. Did family size affect survival?")
                train_tmp = train.copy()
                train_tmp["FamilySize"] = train_tmp["SibSp"] + train_tmp["Parch"] + 1
                family_survival = train_tmp.groupby("FamilySize")["Survived"].mean()
                print(family_survival)


## 4. Interactive EDA
Create interactive Plotly charts (runs in Colab / Jupyter).


In [ ]:
# Survival count
fig = px.bar(train, x="Survived", color="Survived", 
             labels={"Survived": "Survival Status"}, title="Survival Count")
fig.show()

# Survival by sex
fig = px.histogram(train, x="Sex", color="Survived", barmode="group",
                   title="Survival by Gender")
fig.show()

# Sunburst: Pclass -> Sex -> Survived
# Temporary column for better sunburst strings
train_tmp = train.copy()
train_tmp["Survived_Str"] = train_tmp["Survived"].map({0: "Died", 1: "Survived"})
train_tmp["Pclass_Str"] = train_tmp["Pclass"].map({1: "1st Class", 2: "2nd Class", 3: "3rd Class"})

fig = px.sunburst(train_tmp, path=["Pclass_Str", "Sex", "Survived_Str"], 
                  title="Sunburst Chart: Pclass -> Sex -> Survival")
fig.show()


## 5. Static Plot Export for Website
Export static plots to `public/assets/plots/` using a color-blind-safe palette.


In [ ]:
import os

# Color-blind palette definitions
COLORS = {
    "survived": "#0072B2",
    "not_survived": "#D55E00",
    "female": "#CC79A7",
    "male": "#56B4E9",
    "class_1": "#009E73",
    "class_2": "#E69F00",
    "class_3": "#D55E00",
    "neutral": "#6B7280"
}

# Ensure output folder exists
export_path = "../public/assets/plots/"
os.makedirs(export_path, exist_ok=True)

# Export survival count
plt.figure(figsize=(5, 3.5))
sns.countplot(data=train, x="Survived", hue="Survived", 
              palette=[COLORS["not_survived"], COLORS["survived"]], legend=False)
plt.title("Survival Count")
plt.xticks([0, 1], ["Died", "Survived"])
plt.savefig(os.path.join(export_path, "survival_count.png"), bbox_inches="tight", dpi=150)
plt.close()

# Export survival by sex
plt.figure(figsize=(5, 3.5))
sns.barplot(data=train, x="Sex", y="Survived", hue="Sex",
            palette=[COLORS["male"], COLORS["female"]], errorbar=None, legend=False)
plt.title("Survival Rate by Gender")
plt.savefig(os.path.join(export_path, "survival_rate_by_sex.png"), bbox_inches="tight", dpi=150)
plt.close()
print("✓ Exported plots successfully!")


## 6. Feature Engineering
Engineer custom features: FamilySize, IsAlone, Title, CabinKnown, FareLog, and AgeGroup.


In [ ]:
def feature_engineering(df):
    df_out = df.copy()
    df_out.columns = df_out.columns.str.strip().str.lower()

    df_out["family_size"] = df_out["sibsp"] + df_out["parch"] + 1
    df_out["is_alone"] = (df_out["family_size"] == 1).astype(int)
    df_out["cabin_known"] = df_out["cabin"].notna().astype(int)
    df_out["fare_log"] = np.log1p(df_out["fare"])

    def extract_title(name):
        if not isinstance(name, str):
            return "Mr"
        match = re.search(r",\s*([^.]+)\.", name)
        return match.group(1).strip() if match else "Mr"

    df_out["title"] = df_out["name"].apply(extract_title)
    title_map = {"Mr": "Mr", "Mrs": "Mrs", "Miss": "Miss", "Master": "Master", "Mme": "Mrs", "Ms": "Miss", "Mlle": "Miss"}
    df_out["title"] = df_out["title"].map(title_map).fillna("Rare")

    bins = [0, 12, 18, 30, 50, 80, 120]
    labels = ["Child", "Teenager", "Young Adult", "Adult", "Senior", "Unknown"]
    df_out["age_group"] = pd.cut(df_out["age"], bins=bins, labels=labels[:-1]).astype(str)
    df_out["age_group"] = df_out["age_group"].fillna("Unknown")

    # Fill na for model
    df_out["age"] = df_out["age"].fillna(df_out["age"].median())
    df_out["fare"] = df_out["fare"].fillna(df_out["fare"].median())
    df_out["embarked"] = df_out["embarked"].fillna(df_out["embarked"].mode()[0])
    return df_out

train_eng = feature_engineering(train)
test_eng = feature_engineering(test)
print("✓ Features engineered. Shape:", train_eng.shape)


## 7. Model Training and Comparison
Train classical estimators with Pipeline and ColumnTransformer.


In [ ]:
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "family_size", "is_alone", "title", "cabin_known"]
X = train_eng[features]
y = train_eng["survived"].astype(int)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_features = ["age", "sibsp", "parch", "fare", "family_size"]
cat_features = ["pclass", "sex", "embarked", "is_alone", "title", "cabin_known"]

num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, num_features),
    ("cat", cat_transformer, cat_features)
])

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, solver="liblinear", random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
}

results = []
for name, clf in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("classifier", clf)])
    pipe.fit(X_train, y_train)
    val_pred = pipe.predict(X_val)

    acc = accuracy_score(y_val, val_pred)
    prec = precision_score(y_val, val_pred)
    rec = recall_score(y_val, val_pred)
    f1 = f1_score(y_val, val_pred)

    # CV score
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    cv_mean = cv_scores["test_score"].mean()

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "CV Mean": cv_mean
    })

results_df = pd.DataFrame(results)
display(results_df)


## 8. Export Kaggle Submission
Deploy the best model to predict on the test file.


In [ ]:
best_model_name = results_df.sort_values(by="Accuracy", ascending=False).iloc[0]["Model"]
print("Selected Best Classical Model:", best_model_name)

# Fit full training pipeline
clf = models[best_model_name]
full_pipe = Pipeline([("preprocess", preprocessor), ("classifier", clf)])
full_pipe.fit(X, y)

X_test = test_eng[features]
test_preds = full_pipe.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_preds
})

os.makedirs("../submissions/", exist_ok=True)
submission.to_csv("../submissions/submission_best_classical.csv", index=False)
print("✓ Exported submissions/submission_best_classical.csv successfully!")
